In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
 
 
def build_domain_metric_table(df, metric="accuracy"):
 
    """
    Build table:
        rows   -> (fs_method, model)
        cols   -> domains + mean + std
    """
 
    metric_functions = {
        "accuracy": accuracy_score,
        "f1": lambda y_true, y_pred: f1_score(y_true, y_pred, average="binary"),
        "precision": lambda y_true, y_pred: precision_score(y_true, y_pred, average="binary"),
        "recall": lambda y_true, y_pred: recall_score(y_true, y_pred, average="binary")
    }
 
    if metric not in metric_functions:
        raise ValueError(f"Metric {metric} not supported")
 
    metric_fn = metric_functions[metric]
 
    rows = []
 
    grouped = df.groupby(["fs_method", "model"])
 
    for (fs_name, model_name), group in grouped:
 
        domain_scores = {}
 
        for domain_id, domain_group in group.groupby("domain"):
 
            score = metric_fn(
                domain_group["true_label"],
                domain_group["pred_label"]
            )
 
            domain_scores[domain_id] = score
 
        row = {
            "fs_method": fs_name,
            "model": model_name,
            **domain_scores
        }
 
        rows.append(row)
 
    table = pd.DataFrame(rows)
 
    # Sort domain columns numerically if possible
    domain_cols = sorted(
        [c for c in table.columns if isinstance(c, (int, np.integer))]
    )
 
    table = table.set_index(["fs_method", "model"])
 
    # Compute mean and std
    table["mean"] = table[domain_cols].mean(axis=1)
    table["std"] = table[domain_cols].std(axis=1)
 
    return table[domain_cols + ["mean", "std"]]

In [ ]:
acc_table = build_domain_metric_table(results_df, metric="accuracy")
f1_table = build_domain_metric_table(results_df, metric="f1")
 
print(acc_table)